## naver_news 테이블의 한 행을 표현하기 위한 클래스 생성

In [14]:
# 네이버 뉴스 응답 데이터를 담을 NaverNews 엔티티 클래스 정의
from datetime import datetime

class NaverNews:

    def __init__(self, id: int, title: str, originallink: str, link: str, description: str, pub_date: str, created_at: datetime = None):
        self.__id = id
        self.__title = title
        self.__originallink = originallink
        self.__link = link
        self.__description = description
        self.__pub_date = pub_date
        self.__created_at = created_at

    @property
    def id(self):
        return self.__id

    @property
    def title(self):
        return self.__title

    # __title 속성에 대한 setter 메소드 예시
    # 아래 주석을 해제하면 naver_news.title = "새 제목"처럼 값을 변경할 수 있다.
    # @title.setter
    # def title(self, value):
    #     self.__title = value

    @property
    def originallink(self):
        return self.__originallink

    @property
    def link(self):
        return self.__link

    @property
    def description(self):
        return self.__description

    @property
    def pub_date(self):
        return self.__pub_date

    @property
    def created_at(self):
        return self.__created_at

    # naver news api 응답 데이터를 NaverNews 객체로 변환하기 위한 클래스 메서드
    @classmethod
    def from_api_item(cls, item:dict):
        return cls(
            id=None,
            title=item.get('title'),
            originallink=item.get('originallink'),
            link=item.get('link'),
            description=item.get('description'),
            pub_date=item.get('pubDate'),
        )

    def __repr__(self):
        # 객체를 print() 했을 때 어떤 값이 들어 있는지 확인하기 위한 문자열 표현 메소드
        return f'NaverNews({self.__id}, {self.__title}, {self.__originallink}, {self.__link}, {self.__description}, {self.__pub_date}, {self.__created_at})'

## .env를 읽어서 환경변수 등록

In [15]:
from dotenv import load_dotenv
import os

load_dotenv()

NAVER_CLIENT_ID = os.getenv("NAVER_CLIENT_ID")
NAVER_CLIENT_SECRET = os.getenv("NAVER_CLIENT_SECRET")

headers = {
    "X-Naver-Client-Id": NAVER_CLIENT_ID,
    "X-Naver-Client-Secret": NAVER_CLIENT_SECRET
}

config = {
    "host": os.getenv("DB_HOST", "localhost"), # (key, default)
    "port": os.getenv("DB_PORT", "3306"),  # 기본포트 3306인 경우 생략가능
    "user": os.getenv("DB_USER", "skn_ai"),
    "password": os.getenv("DB_PASSWORD", "1234"),
    "database": os.getenv("DB_DATABASE", "menudb")
}

if not NAVER_CLIENT_ID or not NAVER_CLIENT_SECRET:
    raise ValueError('NAVER_CLIENT_ID 또는 NAVER_CLIENT_SECRET이 설정되지 않았습니다.')

## 네이버 뉴스 - "인공지능" 관련 최근 뉴스 10개 조회해서 DB에 저장

In [24]:
import requests
from pprint import pprint # 출력 시 공백문자를 이용해 가독성 좋게 출력

url = 'https://openapi.naver.com/v1/search/news.json'

params = {
    'query' : "인공지능",
    'display' : 10,
    'start' : 1,
    'sort' : 'date'
}

# 수집한 news 데이터를 저장할 리스트
naver_news_list:list[NaverNews] = []

try:
    # GET Method == 조회 요청
    response = requests.get(
        url,
        headers=headers,
        params=params, # dict -> 쿼리 스트링 변환(url encoding도 같이 실행됨)
        timeout=10
    )

    # HTTP 상태 코드가 400, 500번대 인 경우 예외 발생
    response.raise_for_status()

    response_code = response.status_code # 상태 코드
    data = response.json() # 응답 데이터(json) -> dict 변환

    # pprint(data)
    # 응답 데이터 중 뉴스 기사 리스트인 'items' 사용
    items = data.get("items", []) # (key, default)

    # pprint(items) #list[dict]

    naver_news_list = [NaverNews.from_api_item(item) for item in items]

    print("response_code: ", response_code)
    print(naver_news_list)

except requests.exceptions.Timeout:
    print("요청 시간 10초 초과")
except ValueError:
    print("응답 데이터가 JSON 형식이 아닙니다")

response_code:  200
[NaverNews(None, 충남 'AI 대전환' 올인…5.8兆 쏟아붓는다, https://www.hankyung.com/article/2026061575971, https://n.news.naver.com/mnews/article/015/0005298916?sid=102, 충청남도가 제조업 중심의 산업 구조를 <b>인공지능</b>(AI) 기반 산업 체계로 바꾸는 ‘충남 AI 대전환’에 속도를 내고 있다. 도는 2035년까지 5조8900억원을 투입해 제조업 AI 전환(AX)과 데이터센터 구축, AI 인재 양성을... , Mon, 15 Jun 2026 17:23:00 +0900, None), NaverNews(None, 신한은행, 체험형 인턴십 운영... 우수 수료자 채용 우대, http://www.thefirstmedia.net/news/articleView.html?idxno=201362, http://www.thefirstmedia.net/news/articleView.html?idxno=201362, 최근 금융권 전반에서 디지털 전환과 <b>인공지능</b>(AI) 활용이 확대됨에 따라 디지털 역량과 AI 활용 능력, 협업 및 커뮤니케이션 역량을 강화하는 실습형 교육을 확대하고 있다. 현업 멘토링 프로그램도 함께 운영해 신입... , Mon, 15 Jun 2026 17:22:00 +0900, None), NaverNews(None, 비즈데이터, 수처리 공정 XAI 시스템으로 K-water 성과공유제 선정, https://www.mt.co.kr/industry/2026/06/15/2026061514414969916, https://n.news.naver.com/mnews/article/008/0005372190?sid=101, 비즈데이터에 따르면, 이번 선정 과제는 '설명가능한 <b>인공지능</b>(XAI) 기반 수처리 약품 공정 의사결정 시스템' 개발이다. 해당 시스템은 경기도 안산시 시흥·반월 정수장에 구축돼 성능 검

In [19]:
import mysql.connector

try:
    with mysql.connector.connect(**config) as conn:
        with conn.cursor() as cursor:
            for naver_news in naver_news_list:
                cursor.execute('''
                    insert into naver_news(
                        title, originallink, link, description, pub_date
                        )
                        values(%s, %s, %s, %s, %s)
                ''', (naver_news.title, naver_news.originallink, naver_news.link,
                      naver_news.description, naver_news.pub_date))

            conn.commit() # 전체 insert 내용 커밋

except mysql.connector.Error as err:
    print("DB 에러: ", err)
    conn.rollback() # 오류 발생 시 전체 rollback

In [20]:
# DB에 저장된 네이버 뉴스 데이터를 조회해 객체 리스트로 변환
try:
    with mysql.connector.connect(**config) as conn:
        with conn.cursor() as cursor:
            cursor.execute('''
                           select *
                           from naver_news
                           order by id desc
                           ''')  # sql, params
            naver_news_list = [NaverNews(*row) for row in cursor.fetchall()]
            print(naver_news_list)
except mysql.connector.Error as e:
    print('DB 오류:', e)

[NaverNews(10, 콘진원, 'K-컬처 확산' 새 선장 맞았다... 김윤지 원장 본격 행보, http://www.newsworker.co.kr/news/articleView.html?idxno=431361, http://www.newsworker.co.kr/news/articleView.html?idxno=431361, 이를 위한 과제로는 콘텐츠 지식재산(IP)의 새로운 가치 창출 기능 확대와 콘텐츠산업의 전략적 수출 지원, 창작자와 기업이 성장할 수 있는 산업 기반 마련, <b>인공지능</b>(AI) 등 첨단기술에 대한 선제 대응을 제시했다. 단순... , Mon, 15 Jun 2026 16:36:00 +0900, 2026-06-15 16:52:29), NaverNews(9, 한컴, 폴란드 R&amp;D 기업과 잇달아 맞손…유럽 에이전틱 OS 시장 공략, https://www.pinpointnews.co.kr/news/articleView.html?idxno=460282, https://www.pinpointnews.co.kr/news/articleView.html?idxno=460282, 사진=한글과컴퓨터 한컴이 유럽 주요 <b>인공지능</b>(AI)·연구개발(R&amp;D) 기업과 잇따라 손잡고 에이전틱 OS 시장 공략에 속도를 낸다. 한글과컴퓨터는 폴란드 국가공인 R&amp;D 센터 7불스와 업무협약(MOU)을 체결하고 차세대... , Mon, 15 Jun 2026 16:36:00 +0900, 2026-06-15 16:52:29), NaverNews(8, 미래인테크놀러지, AI 기반 HR 인사관리 특허 5건 확보, https://www.mt.co.kr/industry/2026/06/15/2026061512322731091, https://n.news.naver.com/mnews/article/008/0005372141?sid=101, 이번 특허는 △인사평가 △성과관리 △AI(<b>인공지능</b>) 기반 데이터 분석 △자동화 룰 엔진 △HR 워크플

## 중복 뉴스 방지
- 중복 뉴스(중복 행)를 방지하기 위해서는 중복 기준이 되는 컬럼을 설정하고, 컬럼 값이 중복이면 update가 수행되게 해야함.

- 중복 판단을 하려면 PK 또는 UNIQUE 제약조건을 이용
    - 빠른 중복 판단 가능

- `replace` 대신 `on duplicate key update` 구문 사용
    - replace : delete -> insert 형식
    - on duplicate key update : update 1회


```sql
alter table naver_news
add constraint uq_naver_news_link unique (link);
```

In [25]:
insert_count = 0
skip_count = 0

try:
    with mysql.connector.connect(**config) as conn:
        with conn.cursor() as cursor:
            for naver_news in naver_news_list:
                cursor.execute('''
                    insert into naver_news
                        (title, originallink, link, description, pub_date)
                    values
                        (%s, %s, %s, %s, %s)
                    on duplicate key update
                        link = link
                ''', (
                    naver_news.title,
                    naver_news.originallink,
                    naver_news.link,
                    naver_news.description,
                    naver_news.pub_date
                ))

                if cursor.rowcount == 1:
                    insert_count += 1
                else:
                    skip_count += 1

            conn.commit()

    print(f"신규 저장: {insert_count}건")
    print(f"중복 제외: {skip_count}건")

except mysql.connector.Error as e:
    print('DB 오류:', e)

신규 저장: 0건
중복 제외: 10건
